# 03 · Retention & revenue concentration

**Pause-aware active retention** and how revenue concentrates in a small core of repeat customers.

Supports deck slides **2, 6, A2**.

**Retention definition.** A customer is retained at tenure *k* if their last observed delivery is ≥ *k* weeks after acquisition, so a customer who paused and came back still counts. **Censoring:** at tenure *k* the denominator only includes customers with ≥ *k* weeks of observation before 2025-W52.

![Retention and concentration](../reports/figures/s6_retention.png)

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from common import load, customers, weekly_new, CHANNEL_ORDER, LAST_WEEK_IDX
pd.set_option("display.width", 140)
df = load()
c = customers(df)

In [2]:
def curve(sub, kmax=52):
    out = []
    for k in range(kmax + 1):
        eligible = sub[sub.obs_weeks >= k]
        out.append(100 * (eligible.last_tenure >= k).mean())
    return out

overall = curve(c)
cohort = c[c.a_idx <= 12]          # 2023-W01..W13 = 2023-Q1
coh = curve(cohort)
print(f"2023-Q1 cohort n = {len(cohort)}")
for k in (1, 4, 12, 26, 52):
    print(f"week {k:>2}: overall {overall[k]:5.1f}%   2023-Q1 cohort {coh[k]:5.1f}%")
print(f"Never reorder (exactly 1 delivery): {(c.n_deliv == 1).mean() * 100:.1f}% of customers")

2023-Q1 cohort n = 368
week  1: overall  69.1%   2023-Q1 cohort  76.6%
week  4: overall  27.9%   2023-Q1 cohort  33.7%
week 12: overall  11.4%   2023-Q1 cohort  14.1%
week 26: overall   6.1%   2023-Q1 cohort   9.0%
week 52: overall   2.4%   2023-Q1 cohort   2.7%
Never reorder (exactly 1 delivery): 31.1% of customers


## Revenue concentration by lifetime tier

Tiers use each customer's total number of deliveries observed (1 / 2–4 / 5–12 / 13+). The cut-offs are a judgement call, not derived from the data.

![Revenue concentration](../reports/figures/s6_concentration.png)

In [3]:
c["tier"] = pd.cut(c.n_deliv, [0, 1, 4, 12, 10**6], labels=["1 order", "2-4", "5-12", "13+"])
t = c.groupby("tier", observed=True).agg(customers=("net", "size"), net_revenue=("net", "sum"))
t["% of customers"] = (100 * t.customers / t.customers.sum()).round(1)
t["% of net revenue"] = (100 * t.net_revenue / t.net_revenue.sum()).round(1)
t["net_revenue"] = t.net_revenue.round(0)
t

,customers,net_revenue,% of customers,% of net revenue
tier,,,,
1 order,2822,171047.0,31.1,3.7
2-4,3854,885213.0,42.5,19.3
5-12,1460,891404.0,16.1,19.5
13+,926,2632596.0,10.2,57.5


In [4]:
print(f"Total net revenue: ${df.net_revenue.sum():,.0f}")
rep = df[df.is_first_delivery == 0].net_revenue.sum() / df.net_revenue.sum()
print(f"Repeat orders: {rep * 100:.1f}% of net revenue · first orders: {(1 - rep) * 100:.1f}%")

Total net revenue: $4,580,259
Repeat orders: 87.7% of net revenue · first orders: 12.3%
